## OpenAI에서 지원하는 GPT API 사용 실습

## Open AI API 키 발급

- https://platform.openai.com/docs/overview 접속
- 회원가입 / 로그인 진행

### 결제 정보 등록
- 우측 상단 톱니바퀴를 눌러 왼쪽 \[Billing\] 탭으로 이동
- Add payment details로 신용카드 등록
- $5부터 결제할 수 있고, 자동충전 설정은 Off
- \[Limits\] 탭에서 사용한도를 설정 가능

### API Key 생성
- 우측 상단 \[Your Profile\]에서 \[API Keys\]로 들어간 뒤, 새 API key를 생성
- 개인 키를 모든 권한을 부여하여 생성하고 Default project를 사용하여 생성 
    - Secret Key는 노출되지 않도록 잘 관리 (노출시 다른사람이 사용해도 내 지갑에서 결제됨)

---

`.env` 파일을 생성한 뒤, 내용에 다음을 입력

```txt
OPENAI_API_KEY=발급받은 API key를 입력
```

In [1]:
!python --version

Python 3.14.8


In [ ]:
%pip install openai
%pip install python-dotenv
%pip install pydantic

### API Key 설정
- API Key는 보안상 겉으로 보이지 않는 것이 좋으므로 코드상에 노출되지 않도록 하여 사용

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
# OPEN AI API 키 설정
load_dotenv()

True

In [2]:
import os

MY_API_KEY = os.environ['OPENAI_API_KEY']

### `OpenAI` 객체 생성

In [3]:
client = OpenAI()

### Open AI가 지원하는 모델

- https://platform.openai.com/docs/models

In [4]:
OPENAI_MODELS = {
    "gpt-6-astra": {
        'price': {
            'input': 10.00, # per 1M tokens
            'cached': 1.00,
            'output': 50.00,
        }
    },
    "gpt-6-sol": {
        'price': {
            'input': 2.00,
            'cached': 0.20,
            'output': 10.00,
        }
    },
    "gpt-6-luna": { # 가장 저렴한 모델
        'price': {
            'input': 0.10,
            'cached': 0.01,
            'output': 0.50,
        }
    },
    "gpt-5.6-sol": {
        'price': {
            'input': 4.00,
            'cached': 0.40,
            'output': 20.00,
        }
    },
    "gpt-5.6-terra": { 
        'price': {
            'input': 2.00,
            'cached': 0.20,
            'output': 12.00,
        }
    },
    "gpt-5.6-luna": {
        'price': {
            'input': 0.20,
            'cached': 0.02,
            'output': 1.20,
        }
    },
}

MODEL_NAMES = list(OPENAI_MODELS.keys())

In [5]:
MODEL_NAMES

['gpt-6-astra',
 'gpt-6-sol',
 'gpt-6-luna',
 'gpt-5.6-sol',
 'gpt-5.6-terra',
 'gpt-5.6-luna']

In [6]:
MODEL = "gpt-6-luna"

### 첫 API 호출 — Responses API

- `model` : 사용할 모델 이름
- `input` : 프롬프트 (문자열 또는 메시지 목록)
- `output_text` : 최종 텍스트 응답만 추출

In [25]:
question = '생성형 AI를 한 문장으로 설명해줘.'

response = client.responses.create(model = MODEL,
                                    input=question,
                                    )


In [ ]:
response

In [26]:
type(response)

openai.types.responses.response.Response

In [9]:
response.output_text

'생성형 AI는 학습한 데이터를 바탕으로 글, 이미지, 음악, 코드 등 새로운 콘텐츠를 만들어 내는 인공지능입니다.'

### `Response` 객체
- `id`: 응답 고유 id값
- `model`: 실제 사용된 모델
- `status`: 상태 정보
- `usage`에 토큰 사용량 정보가 담김 (`input_tokens`는 입력 토큰의 수, `output__tokens`는 출력 토큰의 수)

In [10]:
print("id     :", response.id)      # 응답 고유 ID (대화 이어가기에 사용)
print("model  :", response.model)   # 실제 사용된 모델
print("status :", response.status)  # completed / incomplete ...

usage = response.usage
print("input_tokens  :", usage.input_tokens)   # 입력 토큰 수
print("output_tokens :", usage.output_tokens)  # 출력 토큰 수 (추론 토큰 포함)
print("total_tokens  :", usage.total_tokens)

id     : resp_039374d11356c69f006ad9780cf26487d0a009606b756a4f2a
model  : gpt-6-luna
status : completed
input_tokens  : 19
output_tokens : 35
total_tokens  : 54


In [11]:
# 전체 구조를 JSON으로 확인
print(response.model_dump_json(indent=2))

{
  "id": "resp_039374d11356c69f006ad9780cf26487d0a009606b756a4f2a",
  "access_programs": null,
  "created_at": 1792636940.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-6-luna",
  "object": "response",
  "output": [
    {
      "id": "msg_039374d11356c69f006ad9780de4f087d0aeb0436c179bef06",
      "content": [
        {
          "annotations": [],
          "text": "생성형 AI는 학습한 데이터를 바탕으로 글, 이미지, 음악, 코드 등 새로운 콘텐츠를 만들어 내는 인공지능입니다.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": "final_answer"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 0.98,
  "background": false,
  "completed_at": 1792636942.0,
  "conversation": null,
  "max_output_tokens": null,
  "max_tool_calls": null,
  "moderation": null,
  "previous_response_id": null,
  "pro

### 비용 계산

- usage × 단가 = 호출 1회 비용

In [12]:
# USD per 1M tokens (Standard, short context, 2026.09) — Docs › Pricing 에서 최신 가격 확인
PRICE = {
    "gpt-6-astra": {"input": 10.00, "cached": 1.00, "output": 50.00},
    "gpt-6-sol":   {"input": 2.00,  "cached": 0.20, "output": 10.00},
    "gpt-6-luna":  {"input": 0.10,  "cached": 0.01, "output": 0.50},
}
USD_KRW = 1350  # 환율 가정

def cost_usd(resp, model=MODEL):
    p = PRICE[model]
    u = resp.usage
    cached = 0
    if getattr(u, "input_tokens_details", None) is not None:
        cached = u.input_tokens_details.cached_tokens or 0
    return (
        (u.input_tokens - cached) * p["input"]
        + cached * p["cached"]
        + u.output_tokens * p["output"]
    ) / 1_000_000

r = client.responses.create(model=MODEL, input="AI 뉴스 3줄 요약 형식 예시를 보여줘")
c = cost_usd(r)
print(f"{c:.6f} USD  (약 {c * USD_KRW:.2f}원)")

0.000113 USD  (약 0.15원)


### 역할(Role) 부여
- instructions (system / developer 메시지와 같은 역할)

In [13]:
response = client.responses.create(
    model=MODEL,
    instructions=(
        "당신은 미국 주식 투자 전문가입니다. "
        "공정한 저널리스트처럼 근거를 들어 보도자료 형태로, "
        "5줄 이내로 응답하세요."
    ),
    input="엔비디아 주식 지금 투자해도 되나요?",
)
print(response.output_text)

**엔비디아 투자 판단: 성장성은 강하지만 가격·변동성 점검 필요**  
AI 가속기와 데이터센터 수요는 성장 동력이지만, 기대가 주가에 이미 반영됐을 가능성도 있습니다.  
경쟁 심화, 수출 규제, 고객사의 자체 칩 개발은 주요 위험 요인입니다.  
따라서 최신 실적과 밸류에이션을 확인하고, 분할 매수·분산 투자 여부를 검토하는 편이 합리적입니다.  
실시간 시세를 반영한 판단은 아니며, 투자 기간과 손실 감내 수준에 따라 결론이 달라집니다.


In [14]:
# instructions만 바꿔서 결과 비교
response = client.responses.create(
    model=MODEL,
    instructions="당신은 초등학생에게 설명하는 친절한 선생님입니다. 쉬운 비유를 들어 3문장으로 답하세요.",
    input="엔비디아 주식 지금 투자해도 되나요?",
)
print(response.output_text)

엔비디아는 성장 기대가 큰 회사지만, 주가는 크게 오르내릴 수 있어서 지금 사는 게 꼭 좋은 선택이라고 단정할 수는 없어요.  
주식 투자는 날씨처럼 예측이 어려우니, 최신 실적과 가격을 살피고 잃어도 괜찮은 돈만 생각해 보세요.  
한 회사에 모든 돈을 넣기보다 여러 곳에 나누어 투자하고, 필요하면 보호자나 금융 전문가와 상의하는 게 좋아요.


### GPT 모델을 설정한 뒤 일반적인 질의 - messages 리스트로 역할 명시

- `OpenAI객체`.chat.completions.create : openAI의 ChatAPI(대화형 인터페이스 제공)를 생성해주는 함수 (`ChatCompletion` 객체 반환)

- model(`str`): 사용할 모델을 설정
- messages(`List\[Dict\]`): role, content를 key로 여러 prompt를 전달할 수 있음
    - role: user, system, assistant
    - ChatGPT 웹 서비스에서는 사용자는 user로 인지하고 입력된 질문이 content로 인식됨
    - system에는 사용자의 질의가 입력되기 전에 모델에게 숙지시켜놓는 대전제나 규칙 등을 지정할 수 있음

In [29]:
question = "안녕? 오늘 피크닉가기 좋은 날씨네!"

response = client.chat.completions.create(model = MODEL,
                                          messages= [
                                              {'role': 'user', 'content': question},
                                              ],
                                        #   max_tokens=150,
                                          reasoning_effort='none',
                                          temperature=0

                                         )


In [30]:
response

ChatCompletion(id='chatcmpl-EbePtPdwvrfDdqU6j1TMNAlujtENL', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='안녕! 그러게, 날씨가 좋으면 피크닉 가기 딱이지 😊 어디로 갈 생각이야?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1792642205, model='gpt-6-luna', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=30, prompt_tokens=19, total_tokens=49, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=0, image_tokens=None, text_tokens=None)))

In [31]:
type(response)

openai.types.chat.chat_completion.ChatCompletion

### `ChatCompletion` 객체
- `id`: 응답 고유 id값
- `choices[0].message`: 응답 메시지 객체 (`ChatCompletionMessage`)
- `ChatCompletionMessage` 객체에 `role`, `content`필드가 있으며, `content`에 실제 메시지가 담겨 있음.
- `response.usage`에 토큰 사용량 정보가 담김 (`prompt_tokens`는 input 토큰의 수, `completion_tokens`는 output 토큰의 수)

In [32]:
response.id   # 문자열 형태

'chatcmpl-EbePtPdwvrfDdqU6j1TMNAlujtENL'

In [33]:
response.usage

CompletionUsage(completion_tokens=30, prompt_tokens=19, total_tokens=49, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=0, image_tokens=None, text_tokens=None))

In [34]:
response.choices[0].message.role

'assistant'

In [35]:
response.choices[0].message.content

'안녕! 그러게, 날씨가 좋으면 피크닉 가기 딱이지 😊 어디로 갈 생각이야?'

### messages 활용 (대화 이력을 직접 리스트로 관리)

- messages를 활용하면 system prompt도 입력이 가능하며, AI의 응답도 기록해둘 수 있음 (이전 대화내용을 기억하도록 하는 것이 가능)

In [36]:
completion = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "당신은 친절한 IT 강사입니다."},
        {"role": "user", "content": "API가 뭐예요?"},
    ],
)
print(completion.choices[0].message.content)

API는 **서로 다른 프로그램이 정보를 주고받거나 기능을 요청할 수 있도록 정해 둔 약속과 연결 방식**이에요.  
영어 *Application Programming Interface*의 약자입니다.

식당에 비유하면, 손님이 주방에 직접 들어가지 않고 **메뉴를 보고 주문하면**, 직원이 주문을 주방에 전달하고 음식을 가져다주죠. API는 이때의 **메뉴와 주문 절차** 같은 역할을 합니다.

예를 들어 날씨 앱은 날씨 정보를 직접 측정하는 대신, 날씨 서비스의 API에 “서울의 현재 날씨를 알려 줘”라고 요청하고 결과를 받아 화면에 보여줄 수 있어요.

즉, API를 이용하면 프로그램끼리 정해진 규칙에 따라 편리하게 기능과 데이터를 주고받을 수 있습니다.


In [37]:
# GPT에게 질문할 텍스트
question = "안녕? 너의 이름이 뭐야?"

response = client.chat.completions.create(model = "gpt-6-luna",
                                          messages = [
                                              {'role': 'user', 'content': '너의 이름은 제이슨이야'},
                                              {'role': 'assistant', 'content': '알겠습니다. 제 이름은 제이슨입니다.'},
                                              {'role': 'user', 'content': question},
                                              ],
                                        #   max_tokens=150,
                                        #   temperature=0
                                         )

In [38]:
response.choices[0].message.content

'안녕! 나는 제이슨이야.'

### 멀티턴 대화

- LLM의 각 호출은 독립적(Stateless) → 이전 대화를 기억하려면 이력을 함께 보내야 함

In [ ]:
# 이전 응답 ID로 이어가기
r1 = client.responses.create(model=MODEL, input="내 이름은 소신이야.")
print("1차:", r1.output_text)

r2 = client.responses.create(
    model=MODEL,
    previous_response_id=r1.id,
    input="내 이름이 뭐라고 했지?",
)
print("2차:", r2.output_text)

### Parameters

- `max_output_tokens`(`int`): 최대 토큰 수 (출력 길이 제한)
- `temperature`(`float`): 다음 단어 생성에 필요한 단어의 확률 분포 자체를 조절 (낮을수록 일관 · 정확, 높을수록 다양 · 창의. 0~2)
- `top_p`(`float`): 단어 확률 분포 중 상위 p% 내의 단어를 사용 (0~1사이 실수, 높을수록 다양한 단어 사용, temperature와 둘 중 하나만 조절 권장)
- `reasoning={"effort": ...}` : 추론 깊이 (none / low / medium / high / xhigh / max)

In [ ]:
for t in [0.0, 1.0, 1.6]:
    r = client.responses.create(
        model=MODEL,
        input="카페 이름을 하나 지어줘. 이름만 답해.",
        reasoning={"effort": "none"},
        temperature=t,
        max_output_tokens=50,
    )
    print(t, "→", r.output_text)

In [ ]:
# reasoning effort 비교: 출력 토큰(추론 토큰 포함)과 응답 시간 차이 관찰
import time

question = "사과 23개 중 20개를 쓰고 7개를 더 샀다. 절반을 팔고 3개를 먹었다면 몇 개 남았나?"
for effort in ["none", "low", "high"]:
    t0 = time.time()
    r = client.responses.create(model=MODEL, input=question, reasoning={"effort": effort})
    print(f"[{effort}] {time.time() - t0:.1f}s / out_tok={r.usage.output_tokens}")
    print(r.output_text, "\n")

### 비교
| | Responses API | Chat Completions |
|---|---|---|
| 입력 | input (+ instructions) | messages |
| 출력 | output_text | choices[0].message.content |
| 대화 상태 | 서버 저장 가능 | 직접 관리 |

### 스트리밍 출력 — 글자가 흘러나오듯 출력 (체감 대기 시간 감소)

In [ ]:
stream = client.responses.create(
    model=MODEL,
    input="생성형 AI의 장점 3가지를 설명해줘.",
    stream=True,
)
for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
print()

### 구조화 출력 (Structured Outputs)
- 스키마를 지정하면 모델이 그 형식을 지켜 응답 → 파싱 에러 없이 바로 사용

In [ ]:
from pydantic import BaseModel


class ProsCons(BaseModel):
    pros: list[str]
    cons: list[str]


r = client.responses.parse(
    model=MODEL,
    input="인공지능(AI)의 장점 3가지와 단점 3가지를 각각 한 문장씩 정리해줘.",
    text_format=ProsCons,
)
result = r.output_parsed
print("장점:", result.pros)
print("단점:", result.cons)

#### 에러 처리

| 에러 | 원인 |
|---|---|
| 401 AuthenticationError | Key 오타 · 미설정 · 폐기된 키 |
| 429 RateLimitError | 요청 과다 또는 잔액 부족 |
| 400 BadRequestError | 지원하지 않는 파라미터 등 |

- InvalidRequestError: 잘못된 요청이 발생한 경우
- APIConnectionError: API 서버와의 연결 문제로 발생
- OpenAIError: 기타 일반적인 openai라이브러리 사용시 발생하는 에러



In [ ]:
# 예외처리를 적용한 함수
import openai

def ask(prompt, retries=3):
    for i in range(retries):
        try:
            return client.responses.create(model=MODEL, input=prompt)
        except openai.RateLimitError:
            wait = 2 ** i
            print(f"Rate limit — {wait}초 후 재시도")
            time.sleep(wait)          # 지수 백오프
        except openai.AuthenticationError:
            raise SystemExit("API Key를 확인하세요")
        except openai.BadRequestError as e:
            print("요청 오류:", e)
            return None
    raise RuntimeError("재시도 초과")

r = ask("안녕하세요!")
if r:
    print(r.output_text)

### Moderation — 유해 콘텐츠 사전 필터링

In [ ]:
mod = client.moderations.create(
    model="omni-moderation-latest",
    input="오늘 날씨 좋네요.",
)
print("flagged:", mod.results[0].flagged)

## 예전 코드

### 프로그래밍 기술적인 내용 질의

In [ ]:
question = "Pandas DataFrame에서 중복 항목을 제거하는 방법은?"

response = client.chat.completions.create(model = "gpt-6-luna",
                                          messages= [{'role': 'user', 'content': question}],
                                          max_tokens=150
                                          )

response.choices[0].message.content

In [ ]:
# 일반 응답에는 이스케이프 코드까지 다 표시되므로 보기 좋게 프린트문으로 출력
print(response.choices[0].message.content)

- OpenAI API는 따로 메모리를 끊어줄 필요가 없음(자동으로 실행 후 끊어짐)
- del client 코드로 openAI API를 담아준 client 객체를 삭제할 수는 있음

### 데이터 시각화 관련 질의

In [ ]:
question = "지난 1년간의 판매 데이터를 시각화하기 위한 막대 차트를 생성해줘"

response = client.chat.completions.create(model = "gpt-6-luna",
                                          messages= [{'role': 'user', 'content': question}],
                                          max_tokens=150,
                                          temperature=0.0
                                         )

print(response.choices[0].message.content)

In [ ]:
# 상위 모델의 성능이 더 좋음
response2 = client.chat.completions.create(model = "gpt-6-sol",
                                           messages= [{'role': 'user', 'content': question}]
                                          )

print(response2.choices[0].message.content)
# ChatGPT 서비스를 이용하는 것이 아니라 쥬피터노트북에서 동작시키는 것이므로 직접 시각화가 되어서 나타나진 않고 코드로 제공함

### 일반적인 분석 질의

In [ ]:
question = "OTT 구독 서비스에서 고객 이탈에 영향을 미치는 주요 요인은?"

response = client.chat.completions.create(model = "gpt-6-luna",
                                          messages= [{'role': 'user', 'content': question}]
                                         )

print(response.choices[0].message.content)